# COOTEFOO — Data Preparation
### Dai 3 knowledge graph (FILAH, TROUT, journalist) alle tabelle finali per il frontend Vue/D3

Questo notebook riassume tutte le decisioni prese durante l'analisi esplorativa e genera
le tabelle finali in formato JSON (una per file, array di oggetti — pronte per
`fetch()`/`d3.json()`), con chiavi primarie/esterne e flag di provenienza `in_filah`/`in_trout`.

**Decisioni incorporate:**
- fix dei type mancanti (Sean, Bay Harvest Corporation, Harbor Odyssey Tours, il place
  `10803677425`, i 2 plan "nudi"), inferiti dagli archi collegati — mai dal solo nome
- merge Discussion + Plan → `Initiative`, con deduplicazione del sentiment quando le due
  fonti coincidono (verificato: succede nella stragrande maggioranza dei casi)
- `TRIP_STOPS` invece di origine/destinazione fissa (un trip può avere fino a 12 tappe)
- flag `in_filah`/`in_trout` binari su ogni tabella anagrafica
- flag `in_filah`/`in_trout` **per singola riga** su `initiative_participants` (non solo
  aggregati sull'iniziativa) — necessario per i pochi casi dove l'iniziativa è nota solo
  parzialmente (es. la discussion è presente ma il plan collegato manca)
- normalizzazione di un'inconsistenza di casing trovata sullo `status`
  (`'Completed'` vs `'completed'`)


## 1. Setup e caricamento dei grafi

In [14]:
import json
from pathlib import Path
from collections import defaultdict

import networkx as nx
import pandas as pd


FILES = {
    "FILAH": "../data/raw_data/FILAH.json",
    "TROUT":  "../data/raw_data/TROUT.json",
    "journalist": "../data/raw_data/journalist.json",
}



def load_graph(path):
    with open(path) as f:
        data = json.load(f)
    return nx.node_link_graph(data, link="links")


## 2. Fix dei type mancanti e correzioni sicure

Recap delle correzioni: `Sean`/`Bay Harvest Corporation`/`Harbor Odyssey Tours` avevano
solo l'`id`, nessun `type` — inferito dagli archi collegati (`travel`/`refers_to` → place,
`about` in entrata → plan, altrimenti override manuale). Le correzioni sui valori (nome di
Sean, coordinate del place recuperate da `road_map.json`, titoli derivati per i 2 plan senza
descrizione) sono tutte "sicure": mai inventate, sempre recuperate da un'altra fonte fornita
o puramente cosmetiche e marcate come tali (`type_inferred`, `title_is_derived`).


In [15]:
MANUAL_TYPE_OVERRIDES = {
    "Bay Harvest Corporation": "entity.organization",
    "Harbor Odyssey Tours": "entity.organization",
    "Sean": "entity.person",
}


def infer_and_fix_types(G):
    for node_id, data in G.nodes(data=True):
        if "type" in data:
            continue
        if node_id in MANUAL_TYPE_OVERRIDES:
            inferred = MANUAL_TYPE_OVERRIDES[node_id]
        else:
            in_roles = {d.get("role") for _, _, d in G.in_edges(node_id, data=True)}
            if in_roles & {"travel", "refers_to"}:
                inferred = "place"
            elif "about" in in_roles:
                inferred = "plan"
            else:
                inferred = None
        if inferred:
            G.nodes[node_id]["type"] = inferred
            G.nodes[node_id]["type_inferred"] = True


def apply_safe_corrections(G):
    if "Sean" in G.nodes:
        G.nodes["Sean"]["name"] = "Sean"
        G.nodes["Sean"]["role"] = None
        G.nodes["Sean"]["role_note"] = "non e' uno dei 6 membri ufficiali COOTEFOO - identita' incerta"

    if 10803677425 in G.nodes:
        # coordinate/zona recuperate automaticamente da road_map.json in build_place()
        G.nodes[10803677425].update({
            "name": "Harbor Odyssey Tours",
            "source_of_fix": "road_map.json",
        })

    for nid in ["name_harbor_area_Meeting_11_Harbor_Odyssey_Tours", "concert_Travel_Harborfront_Market"]:
        if nid in G.nodes:
            G.nodes[nid]["short_title"] = nid.replace("_", " ")
            G.nodes[nid]["title_is_derived"] = True


graphs = {name: load_graph(p) for name, p in FILES.items()}
for G in graphs.values():
    infer_and_fix_types(G)
    apply_safe_corrections(G)

G_journ = graphs["journalist"]
G_filah = graphs["FILAH"]
G_trout = graphs["TROUT"]

print("Grafi caricati e corretti:")
for name, G in graphs.items():
    print(f"  {name:12s} nodes={G.number_of_nodes():4d}  edges={G.number_of_edges():4d}")


Grafi caricati e corretti:
  FILAH        nodes= 396  edges= 765
  TROUT        nodes= 164  edges= 378
  journalist   nodes= 740  edges=2436


## 3. Funzioni di utilità (riusate in più tabelle)

In [16]:
def in_sub(node_id):
    return {"in_filah": node_id in G_filah.nodes, "in_trout": node_id in G_trout.nodes}


def nodes_of_type(G, t):
    return {n: d for n, d in G.nodes(data=True) if d.get("type") == t}


def normalize_status(s):
    # fix qualita' dati: trovata inconsistenza di casing 'Completed' vs 'completed'
    return s.lower() if isinstance(s, str) else s


def discussion_plan_links(G):
    return [
        (u, v, normalize_status(d.get("status")))
        for u, v, d in G.edges(data=True)
        if d.get("role") == "about" and G.nodes.get(v, {}).get("type") == "plan"
    ]


def meeting_of_map(G):
    m = {}
    for u, v, d in G.edges(data=True):
        if d.get("role") == "part_of":
            m[v] = u
    return m


def participant_full_map(G, node_id):
    m = {}
    if node_id in G.nodes:
        for _, v, d in G.out_edges(node_id, data=True):
            if d.get("role") == "participant":
                m[v] = (d.get("sentiment"), d.get("reason"), d.get("industry"))
    return m


## 4. Tabelle anagrafiche: PERSON, ORGANIZATION, PLACE, TOPIC, MEETING

**Nota su `PLACE` e `road_map.json`:** incrociando i place di `journalist` con
`road_map.json` è emerso che il 100% dei 160 place condivide lo stesso `id` con un nodo
della rete stradale. Questo ha permesso di scoprire e correggere un bug: in `journalist`
i campi `lat`/`lon` sono **scambiati** rispetto alla convenzione standard (`lat` riporta
in realtà la longitudine). `road_map.json` usa i nomi corretti ed è stato usato come fonte
di verità per `latitude`/`longitude`, aggiungendo anche `city_name` (assente in
`journalist`). `road_map.json` resta comunque un file a sé nell'output finale — non viene
fuso in `places.json`, che riporta solo il sottoinsieme di luoghi rilevanti per COOTEFOO.
`oceanus_map.geojson` non richiede invece alcuna trasformazione: è già nel formato nativo
per il rendering di mappe in D3.js/Leaflet.


In [17]:
def build_person():
    rows = []
    for nid, d in nodes_of_type(G_journ, "entity.person").items():
        rows.append({
            "id": nid, "name": d.get("name"), "role": d.get("role"),
            "role_note": d.get("role_note"), "type_inferred": d.get("type_inferred", False),
            **in_sub(nid),
        })
    return pd.DataFrame(rows)


def build_organization():
    rows = []
    for nid, d in nodes_of_type(G_journ, "entity.organization").items():
        rows.append({
            "id": nid, "type_inferred": d.get("type_inferred", False), **in_sub(nid),
        })
    return pd.DataFrame(rows)


def load_road_map_index():
    """road_map.json condivide gli stessi id di 'place' in journalist (100% di
    sovrapposizione, verificato) ma usa i nomi corretti latitude/longitude - in journalist
    sono scambiati (il campo 'lat' e' in realta' la longitudine e viceversa) - e aggiunge
    'city_name', assente in journalist."""
    with open("../data/raw_data/road_map.json") as f:
        rm = json.load(f)
    return {n["id"]: n for n in rm["nodes"]}


ROAD_MAP_INDEX = load_road_map_index()


def build_place():
    rows = []
    for nid, d in nodes_of_type(G_journ, "place").items():
        rm_node = ROAD_MAP_INDEX.get(nid, {})
        longitude = rm_node.get("longitude", d.get("lat"))
        latitude = rm_node.get("latitude", d.get("lon"))
        # unico campo 'name': se 'name' manca ma 'label' e' presente, uso 'label'
        # (verificato: quando entrambi presenti, coincidono sempre - 0 mismatch)
        name = d.get("name") if d.get("name") else d.get("label")
        rows.append({
            "id": nid, "name": name, "city_name": rm_node.get("city_name"),
            "latitude": latitude, "longitude": longitude,
            "zone": d.get("zone") if d.get("zone") is not None else rm_node.get("zone"),
            "zone_detail": d.get("zone_detail"),
            "type_inferred": d.get("type_inferred", False), "source_of_fix": d.get("source_of_fix"),
            **in_sub(nid),
        })
    return pd.DataFrame(rows)


def build_topic():
    rows = []
    for nid, d in nodes_of_type(G_journ, "topic").items():
        rows.append({
            "id": nid, "short_topic": d.get("short_topic"), "long_topic": d.get("long_topic"),
            **in_sub(nid),
        })
    return pd.DataFrame(rows)


def build_meeting():
    rows = []
    for nid, d in nodes_of_type(G_journ, "meeting").items():
        rows.append({"id": nid, "date_label": d.get("date"), **in_sub(nid)})
    return pd.DataFrame(rows)


person_df = build_person()
organization_df = build_organization()
place_df = build_place()
topic_df = build_topic()
meeting_df = build_meeting()

print("person:", len(person_df), " organization:", len(organization_df),
      " place:", len(place_df), " topic:", len(topic_df), " meeting:", len(meeting_df))
person_df


person: 7  organization: 10  place: 173  topic: 15  meeting: 16


,id,name,role,role_note,type_inferred,in_filah,in_trout
0,Seal,Seal,Committee Chair,None,False,True,True
1,Ed Helpsford,Ed Helpsford,Vice Chair,None,False,False,True
2,Teddy Goldstein,Teddy Goldstein,Treasurer,None,False,False,True
3,Simone Kat,Simone Kat,Member,None,False,True,True
4,Tante Titan,Tante Titan,Member,None,False,False,True
5,Carol Limpet,Carol Limpet,Member,None,False,True,True
6,Sean,Sean,None,non e' uno dei 6 membri ufficiali COOTEFOO - i...,True,False,False


## 5. Spostamenti: TRIP + TRIP_STOPS

Non origine/destinazione fissa: un trip è una sequenza di tappe (fino a 12), ognuna con il
proprio orario. `TRIP_STOPS` va ordinata per `time` per ricostruire il percorso completo.


In [18]:
def build_trip_and_stops():
    trip_rows, stop_rows = [], []
    for nid, d in nodes_of_type(G_journ, "trip").items():
        person_id = None
        for _, v, ed in G_journ.out_edges(nid, data=True):
            if ed.get("role") is None and G_journ.nodes.get(v, {}).get("type") == "entity.person":
                person_id = v
        trip_rows.append({
            "id": nid, "person_id": person_id, "date": d.get("date"),
            "start": d.get("start"), "end": d.get("end"), **in_sub(nid),
        })
        for _, v, ed in G_journ.out_edges(nid, data=True):
            if ed.get("role") is None and G_journ.nodes.get(v, {}).get("type") == "place":
                stop_rows.append({"trip_id": nid, "place_id": v, "time": ed.get("time")})
    return pd.DataFrame(trip_rows), pd.DataFrame(stop_rows)


trip_df, trip_stops_df = build_trip_and_stops()
print("trip:", len(trip_df), " trip_stops:", len(trip_stops_df))
trip_df.head()


trip: 342  trip_stops: 1363


,id,person_id,date,start,end,in_filah,in_trout
0,trip_0,Simone Kat,0040-04-24,09:00:00,21:00:00,True,False
1,trip_1,Seal,2040-06-16,06:29:00,12:36:00,True,False
2,trip_2,Ed Helpsford,2040-06-28,08:23:00,08:23:00,False,False
3,trip_3,Teddy Goldstein,2040-06-06,07:28:00,07:59:00,False,False
4,trip_4,Seal,2040-07-17,07:50:00,14:42:00,True,False


## 6. Iniziative: INITIATIVE + INITIATIVE_STATUS_TIMELINE + INITIATIVE_PARTICIPANTS

Il cuore della trasformazione: ogni `plan` (più le discussion "orfane" senza plan collegato)
diventa una `Initiative`. I partecipanti sono l'unione deduplicata di discussion+plan, con
flag `in_filah`/`in_trout` calcolati **per singola riga** (persona × iniziativa), non
aggregati sull'iniziativa intera — necessario per i casi in cui l'iniziativa è nota solo
parzialmente in un dataset di parte.


In [19]:
STATUS_ORDER = ["introduced", "planned", "in_progress", "completed"]


def build_initiative_tables():
    dp_links = discussion_plan_links(G_journ)
    plan_discs = defaultdict(list)
    for disc, plan, status in dp_links:
        plan_discs[plan].append((disc, status))

    all_plans = set(nodes_of_type(G_journ, "plan").keys())
    all_discussions = set(nodes_of_type(G_journ, "discussion").keys())
    linked_discussions = {disc for disc, plan, status in dp_links}
    orphan_discussions = all_discussions - linked_discussions

    meeting_of = meeting_of_map(G_journ)

    def single_target(node_id, role, target_type):
        for _, v, d in G_journ.out_edges(node_id, data=True):
            if d.get("role") == role and G_journ.nodes.get(v, {}).get("type") == target_type:
                return v
        return None

    initiative_rows = []
    timeline_rows = []
    participant_rows = []

    def process_initiative(initiative_id, source_type, plan_id, discussions_with_status):
        discussion_ids = [d for d, s in discussions_with_status]

        topic_id = single_target(plan_id, "plan", "topic") if plan_id else None
        if topic_id is None and discussion_ids:
            topic_id = single_target(discussion_ids[0], "about", "topic")

        place_id = single_target(plan_id, "travel", "place") if plan_id else None
        place_id_conflict = None
        for disc in discussion_ids:
            disc_place = single_target(disc, "refers_to", "place")
            if disc_place is not None and place_id is not None and disc_place != place_id:
                place_id_conflict = disc_place

        title_source = G_journ.nodes.get(plan_id, {}) if plan_id else G_journ.nodes.get(initiative_id, {})
        short_title = title_source.get("short_title")
        long_title = title_source.get("long_title")
        plan_type = title_source.get("plan_type")
        title_is_derived = title_source.get("title_is_derived", False)

        created_meeting_id = meeting_of.get(plan_id) if plan_id else meeting_of.get(initiative_id)

        statuses_present = [s for d, s in discussions_with_status if s]
        final_status = max(statuses_present, key=lambda s: STATUS_ORDER.index(s)) if statuses_present else None

        def initiative_present(G_sub):
            if plan_id and plan_id in G_sub.nodes:
                return True
            return any(d in G_sub.nodes for d in discussion_ids)

        initiative_rows.append({
            "id": initiative_id, "source_type": source_type, "topic_id": topic_id,
            "place_id": place_id, "place_id_conflict": place_id_conflict,
            "short_title": short_title, "long_title": long_title, "plan_type": plan_type,
            "title_is_derived": title_is_derived, "created_meeting_id": created_meeting_id,
            "final_status": final_status, "n_checkpoint": len(discussion_ids),
            "in_filah": initiative_present(G_filah), "in_trout": initiative_present(G_trout),
        })

        for disc, status in discussions_with_status:
            timeline_rows.append({
                "initiative_id": initiative_id, "meeting_id": meeting_of.get(disc), "status": status,
            })

        canonical = {}
        for disc in discussion_ids:
            canonical.update(participant_full_map(G_journ, disc))
        if plan_id:
            canonical.update(participant_full_map(G_journ, plan_id))

        for entity_id, (sentiment, reason, industry) in canonical.items():
            entity_type = G_journ.nodes.get(entity_id, {}).get("type", "NO_TYPE")

            def entity_present(G_sub):
                for disc in discussion_ids:
                    if entity_id in participant_full_map(G_sub, disc):
                        return True
                if plan_id and entity_id in participant_full_map(G_sub, plan_id):
                    return True
                return False

            participant_rows.append({
                "initiative_id": initiative_id, "entity_id": entity_id, "entity_type": entity_type,
                "sentiment": sentiment, "reason": reason,
                "industry": industry if industry else [],
                "opinion_recorded": sentiment is not None,
                "in_filah": entity_present(G_filah), "in_trout": entity_present(G_trout),
            })

    for plan_id in all_plans:
        process_initiative(plan_id, "plan", plan_id, plan_discs.get(plan_id, []))

    for disc in orphan_discussions:
        process_initiative(disc, "discussion", None, [(disc, None)])

    return (pd.DataFrame(initiative_rows), pd.DataFrame(timeline_rows), pd.DataFrame(participant_rows))


initiative_df, timeline_df, initiative_participants_df = build_initiative_tables()
print("initiative:", len(initiative_df), " timeline:", len(timeline_df),
      " initiative_participants:", len(initiative_participants_df))
initiative_df.head()


initiative: 79  timeline: 101  initiative_participants: 113


,id,source_type,topic_id,place_id,place_id_conflict,short_title,long_title,plan_type,title_is_derived,created_meeting_id,final_status,n_checkpoint,in_filah,in_trout
0,fish_vacuum_Meeting_1_Introduction,plan,fish_vacuum,NaN,NaN,fish_vacuum_Meeting_1_Introduction,General Introduction to what a fish vacuum is,presentation,False,Meeting_1,completed,1,True,True
1,deep_fishing_dock_Meeting_3_Maintenance_Plan,plan,deep_fishing_dock,NaN,NaN,deep_fishing_dock_Meeting_3_Maintenance_Plan,Discuss maintenance plan and designate travel ...,proposal,False,Meeting_3,planned,1,True,True
2,low_volume_crane_Travel_Harbor_Builders,plan,low_volume_crane,35922181.0,NaN,low_volume_crane_Travel_Harbor_Builders,Travel to Harbor Builders in Haacklee,Travel,False,Meeting_8,completed,2,True,False
3,affordable_housing_Meeting_9_Invite_Developers,plan,affordable_housing,NaN,NaN,affordable_housing_Meeting_9_Invite_Developers,Invite housing developers,feedback,False,Meeting_9,planned,1,True,True
4,marine_life_deck_Meeting_12_Environmental_Impa...,plan,marine_life_deck,NaN,NaN,marine_life_deck_Meeting_12_Environmental_Impa...,Present findings from the environmental impact...,Report,False,Meeting_12,planned,1,False,True


## 7. Verifica di sanità sul caso "grave" già discusso

`name_inspection_office_Meeting_8_Proposal`: il plan manca del tutto in FILAH, ma la sua
discussion è presente. Deve risultare `in_filah=True` sull'iniziativa (OR logic), con
Simone Kat (`in_filah=True`, dalla discussion) e Tante Titan (`in_filah=False`, solo sul
plan mancante) — cioè l'informazione parziale deve restare visibile riga per riga.


In [20]:
check_init = initiative_df[initiative_df.id == "name_inspection_office_Meeting_8_Proposal"]
print(check_init.to_string(index=False))
print()
check_part = initiative_participants_df[
    initiative_participants_df.initiative_id == "name_inspection_office_Meeting_8_Proposal"
]
print(check_part.to_string(index=False))


                                       id source_type               topic_id  place_id  place_id_conflict                               short_title                                long_title plan_type  title_is_derived created_meeting_id final_status  n_checkpoint  in_filah  in_trout
name_inspection_office_Meeting_8_Proposal        plan name_inspection_office       NaN                NaN name_inspection_office_Meeting_8_Proposal Propose potential names based on feedback  proposal             False          Meeting_8      planned             1      True     False

                            initiative_id   entity_id   entity_type  sentiment                                                                  reason industry  opinion_recorded  in_filah  in_trout
name_inspection_office_Meeting_8_Proposal  Simone Kat entity.person        0.0 Symbolic gesture with no direct impact on tourism or vessel operations.       []              True      True     False
name_inspection_office_Meeting_8_Pr

## 8. Export delle tabelle finali in JSON

In [21]:
TABLES = {
    "persons": person_df,
    "organizations": organization_df,
    "places": place_df,
    "topics": topic_df,
    "meetings": meeting_df,
    "trips": trip_df,
    "trip_stops": trip_stops_df,
    "initiatives": initiative_df,
    "initiative_status_timeline": timeline_df,
    "initiative_participants": initiative_participants_df,
}

print("Riepilogo tabelle generate:")
for name, df in TABLES.items():
    print(f"  {name:28s} {len(df):5d} righe   colonne: {list(df.columns)}")

for name, df in TABLES.items():
    out_path = f"../data/{name}.json"
    df.to_json(out_path, orient="records", indent=2)



Riepilogo tabelle generate:
  persons                          7 righe   colonne: ['id', 'name', 'role', 'role_note', 'type_inferred', 'in_filah', 'in_trout']
  organizations                   10 righe   colonne: ['id', 'type_inferred', 'in_filah', 'in_trout']
  places                         173 righe   colonne: ['id', 'name', 'city_name', 'latitude', 'longitude', 'zone', 'zone_detail', 'type_inferred', 'source_of_fix', 'in_filah', 'in_trout']
  topics                          15 righe   colonne: ['id', 'short_topic', 'long_topic', 'in_filah', 'in_trout']
  meetings                        16 righe   colonne: ['id', 'date_label', 'in_filah', 'in_trout']
  trips                          342 righe   colonne: ['id', 'person_id', 'date', 'start', 'end', 'in_filah', 'in_trout']
  trip_stops                    1363 righe   colonne: ['trip_id', 'place_id', 'time']
  initiatives                     79 righe   colonne: ['id', 'source_type', 'topic_id', 'place_id', 'place_id_conflict', 'short_ti